# 2D Convolution Accelerator Test on PYNQ

This notebook tests the `conv2d_stream` HLS IP on the PYNQ board.

It demonstrates:

1. Loading a 3×3 kernel through AXI-Lite registers.
2. Sending a 5×5 image through AXI DMA.
3. Receiving a 7×7 full-convolution output.
4. Comparing the hardware output against a Python reference model.
5. Measuring execution using the AXI Timer.

Upload `conv2d_design.bit` and `conv2d_design.hwh` to the same folder as this notebook before running.


## 1. Imports and overlay loading

In [ ]:
import time
import numpy as np
from pynq import Overlay, allocate

ol = Overlay("conv2d_design.bit")
print("Overlay loaded successfully")
print("IP blocks:")
for name in ol.ip_dict.keys():
    print("  ", name)

dma = ol.axi_dma_0
conv2d = ol.conv2d_stream_0
hw_timer = ol.axi_timer_0


## 2. Problem dimensions and stream lengths

The HLS implementation currently expects 35 input stream words: the 25 real image pixels plus 10 extra zeros corresponding to two padded rows. It produces 49 output stream words for the 7×7 full-convolution output.

In [ ]:
IH = 5
IW = 5
KH = 3
KW = 3
OH = IH + KH - 1   # 7
OW = IW + KW - 1   # 7

N_INPUT_REAL = IH * IW       # 25
N_INPUT_STREAM = 7 * 5       # 35, because the HLS design reads 2 extra zero rows
N_OUTPUT = OH * OW           # 49

print(f"Input image shape     : {IH} x {IW}")
print(f"Kernel shape          : {KH} x {KW}")
print(f"Output image shape    : {OH} x {OW}")
print(f"DMA input words       : {N_INPUT_STREAM}")
print(f"DMA output words      : {N_OUTPUT}")


## 3. AXI-Lite register offsets

These offsets should match the generated HLS driver header, usually found at:

```text
hls_2dconv_stream_pipelined/sol1/impl/ip/drivers/conv2d_stream_v1_0/src/xconv2d_stream_hw.h
```

If your generated offsets differ, update the dictionary below.

In [ ]:
CTRL_REG = 0x00

K_OFFSETS = {
    "k00": 0x10,
    "k01": 0x18,
    "k02": 0x20,
    "k10": 0x28,
    "k11": 0x30,
    "k12": 0x38,
    "k20": 0x40,
    "k21": 0x48,
    "k22": 0x50,
}

def write_kernel(ip, kernel):
    # Write a 3x3 integer kernel to the conv2d_stream AXI-Lite registers.
    kernel = np.asarray(kernel, dtype=np.int32)
    assert kernel.shape == (3, 3)

    ip.write(K_OFFSETS["k00"], int(kernel[0, 0]))
    ip.write(K_OFFSETS["k01"], int(kernel[0, 1]))
    ip.write(K_OFFSETS["k02"], int(kernel[0, 2]))

    ip.write(K_OFFSETS["k10"], int(kernel[1, 0]))
    ip.write(K_OFFSETS["k11"], int(kernel[1, 1]))
    ip.write(K_OFFSETS["k12"], int(kernel[1, 2]))

    ip.write(K_OFFSETS["k20"], int(kernel[2, 0]))
    ip.write(K_OFFSETS["k21"], int(kernel[2, 1]))
    ip.write(K_OFFSETS["k22"], int(kernel[2, 2]))

def start_ip(ip):
    # Start the HLS IP by setting ap_start.
    ip.write(CTRL_REG, 0x01)

def wait_ip_done(ip, timeout_s=1.0):
    # Optional polling of ap_done. DMA wait is usually enough.
    t0 = time.perf_counter()
    while (ip.read(CTRL_REG) & 0x02) == 0:
        if time.perf_counter() - t0 > timeout_s:
            raise TimeoutError("conv2d_stream did not assert ap_done")
    return True


## 4. Python reference model

This reference model matches your HLS implementation: the kernel is assumed to already be in the orientation used by the sliding window. So this model does **not** flip the kernel again.

If you want mathematical convolution, pass an already-flipped kernel into both the HLS IP and this reference model.

In [ ]:
def conv2d_full_preflipped_kernel(image, kernel):
    """
    Reference model matching the HLS implementation.

    The kernel is assumed to already be in the orientation used by the
    sliding-window hardware. This function does not flip the kernel again.
    """
    image = np.asarray(image, dtype=np.int32)
    kernel = np.asarray(kernel, dtype=np.int32)

    padded = np.zeros((OH + 2, OW + 2), dtype=np.int32)
    padded[2:2+IH, 2:2+IW] = image

    out = np.zeros((OH, OW), dtype=np.int32)

    for r in range(OH):
        for c in range(OW):
            window = padded[r:r+KH, c:c+KW]
            out[r, c] = np.sum(window * kernel)

    return out


## 5. Test image, kernel, and expected output

In [ ]:
image = np.array([
    [ 1,  2,  3,  4,  5],
    [ 6,  7,  8,  9, 10],
    [11, 12, 13, 14, 15],
    [16, 17, 18, 19, 20],
    [21, 22, 23, 24, 25],
], dtype=np.int32)

# This kernel should be the already-inverted/sliding-window version expected by your HLS implementation.
kernel = np.array([
    [-1, -2, -3],
    [-4, -5, -6],
    [-7, -8, -9],
], dtype=np.int32)

expected = conv2d_full_preflipped_kernel(image, kernel)

print("Input image:")
print(image)

print("Kernel loaded into HLS:")
print(kernel)

print("Expected output:")
print(expected)


## 6. Allocate DMA buffers

The input buffer contains the 25 real image pixels followed by 10 zeros. The output buffer receives 49 integers.

In [ ]:
in_buf = allocate(shape=(N_INPUT_STREAM,), dtype=np.int32)
out_buf = allocate(shape=(N_OUTPUT,), dtype=np.int32)

# First 25 words are image pixels, row-major.
in_buf[:N_INPUT_REAL] = image.flatten()

# Last 10 words are two extra zero rows, because the HLS design reads 35 words.
in_buf[N_INPUT_REAL:] = 0

out_buf[:] = 0

print("Input DMA buffer:")
print(np.array(in_buf))


## 7. Run the accelerator using DMA

Safe order:

1. Start receive DMA.
2. Start send DMA.
3. Start the HLS IP.

This avoids output-side stalling because the hardware output stream is not an infinite buffer.

In [ ]:
print("--- DMA Streaming 2D Convolution ---")

write_kernel(conv2d, kernel)

t_start = time.perf_counter()

dma.recvchannel.transfer(out_buf)
dma.sendchannel.transfer(in_buf)
start_ip(conv2d)

dma.sendchannel.wait()
dma.recvchannel.wait()

t_dma = time.perf_counter() - t_start

hw_out = np.array(out_buf, dtype=np.int32).reshape(OH, OW)

print(f"DMA transfer + accelerator time: {t_dma * 1e3:.3f} ms")
print("Hardware output:")
print(hw_out)


## 8. Verify correctness

In [ ]:
diff = hw_out - expected
max_abs_diff = np.max(np.abs(diff))

print("Difference:")
print(diff)

print(f"Max absolute difference = {max_abs_diff}")

if max_abs_diff == 0:
    print("PASS: hardware output matches Python reference")
else:
    print("FAIL: hardware output differs from Python reference")


## 9. AXI Timer measurement

In [ ]:
# AXI Timer register map:
#   0x00 TCSR0
#   0x04 TLR0
#   0x08 TCR0

TCSR0 = 0x00
TLR0  = 0x04
TCR0  = 0x08

def timer_start(timer):
    timer.write(TLR0, 0)
    timer.write(TCSR0, 0x020)   # LOAD
    timer.write(TCSR0, 0x080)   # ENT, count up

def timer_stop(timer):
    cycles = timer.read(TCR0)
    timer.write(TCSR0, 0x000)
    return cycles

# Reinitialize buffers.
in_buf[:N_INPUT_REAL] = image.flatten()
in_buf[N_INPUT_REAL:] = 0
out_buf[:] = 0

write_kernel(conv2d, kernel)

dma.recvchannel.transfer(out_buf)
dma.sendchannel.transfer(in_buf)

timer_start(hw_timer)
start_ip(conv2d)

dma.sendchannel.wait()
dma.recvchannel.wait()

cycles = timer_stop(hw_timer)

FCLK_MHZ = 100.0
hw_time_us = cycles / FCLK_MHZ

print(f"HW timer: {cycles} cycles = {hw_time_us:.3f} us @ {FCLK_MHZ:.0f} MHz")


## 10. Cleanup

Run this after finishing all tests. If you want to rerun previous cells, reallocate the buffers by rerunning the allocation cell.

In [ ]:
in_buf.freebuffer()
out_buf.freebuffer()
print("DMA buffers freed")
